In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
#reading all tables from silver
g_vehicle_master = spark.read.table('proj_veh.silver.silver_vehicle_master')
g_maintenance_logs = spark.read.table('proj_veh.silver.silver_maintenance_logs')
g_oem_recalls = spark.read.table('proj_veh.silver.silver_oem_recalls')
g_warranty_claims = spark.read.table('proj_veh.silver.silver_warranty_claims')


In [0]:
#joining
gold_fleet_cost_summary = g_vehicle_master.join(g_maintenance_logs,on = 'vehicle_id',how = 'left')

In [0]:
df = gold_fleet_cost_summary\
                    .groupBy('vehicle_id')\
                    .agg(sum('cost_usd').alias('total_maintenance_cost'),
                        count('vehicle_id').alias('total_service_visit'),
                        avg('cost_usd').alias('avg_cost_per_visit'),
                        max('service_date').alias('last_service_date')
                    ).withColumn('ingestion_timestamp',current_timestamp())


In [0]:
df1.write.format('delta').mode('overwrite').saveAsTable('proj_veh.gold.gold_fleet_cost_summary')

In [0]:
g_vehicle_master.alias('v')
g_oem_recalls.alias('r')

condition = (col('v.make') == col('r.make')) & \
            (col('v.year') >= col('r.year_start')) & \
            (col('v.year') <= col('r.year_end'))

df_n = g_vehicle_master.alias('v').join(g_oem_recalls.alias('r'),on = condition)


In [0]:
gold_recall_risk_register = df_n.select(col('v.vehicle_id'),col('v.make'),col('v.model'),col('v.year'),col('r.recall_id'),col('r.description'),col('r.recall_date')).withColumn('ingestion_timestamp', current_timestamp())

gold_recall_risk_register.write.format('delta').mode('overwrite').saveAsTable('proj_veh.gold.gold_recall_risk_register')

In [0]:
%sql
select count(*)
from proj_veh.gold.gold_recall_risk_register


count(*)
25


In [0]:
veh_master = spark.read.table('proj_veh.silver.silver_vehicle_master')
recall_register = spark.read.table('proj_veh.gold.gold_recall_risk_register')
warranty_claims = spark.read.table('proj_veh.silver.silver_warranty_claims')

In [0]:
veh_master = veh_master.drop('ingestion_timestamp')
recall_register = recall_register.drop('ingestion_timestamp')
warranty_claims = warranty_claims.drop('ingestion_timestamp')

In [0]:
has_recall = veh_master.alias('v').join(
                        recall_register.alias('r'), on='vehicle_id', how='left')

has_claim = has_recall.join(
                        warranty_claims.alias('w'), on='vehicle_id', how='left')

In [0]:
df_r = has_claim.withColumn('risk_score',
                            when((col('recall_id').isNotNull()) & (col('claim_id').isNotNull()), "HIGH")
                            .when((col('recall_id').isNotNull()) | (col('claim_id').isNotNull()), "MEDIUM")
                            .otherwise("LOW"))\
                            .withColumn('ingestion_timestamp',current_timestamp())



In [0]:
df_r.select(
    col('v.vehicle_id'),
    col('v.make'),        
    col('v.model'),       
    col('v.year'),
    col('r.recall_id'),
    col('w.claim_id'),
    'risk_score',
    col('ingestion_timestamp'))\
    .write.format('delta').mode('overwrite')\
    .saveAsTable('proj_veh.gold.gold_high_risk_vehicles')

In [0]:
df_r2 = df_r.select('risk_score')
df_r2.display()

risk_score
HIGH
HIGH
MEDIUM
HIGH
HIGH
HIGH
HIGH
HIGH
HIGH
HIGH


In [0]:
%sql
select count(*)
from proj_veh.gold.gold_high_risk_vehicles

count(*)
38
